# 🎯 [PRC07_M06] 부품 조달 위험 공급망 지도 및 대체선 추천 인터랙티브 대시보드 구현 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC07/prc07_m06_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC07_M06 |
| **미션 주제** | 부품 조달 위험 공급망 지도 및 대체선 추천 인터랙티브 대시보드 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
무기체계 부품은 해외 의존도가 높고 공급선이 하나뿐이거나 단종(DMSMS) 공고가 나오면 조달이 막히기 쉬운데, 위험 신호를 사람이 문서와 전화로 모으면 국산화 후보를 찾기까지 수개월이 걸린다. 본 프로젝트는 부품 공급망 자료에서 조달 병목과 위험 신호를 찾고 국산화 대체 후보를 자동으로 추천하는 도구를 만드는 과제로, 업무 단계별 병목과 위험 신호 기준, 추천 수정안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 부품 100종과 가상 국가·가상 업체 자료로 공급망 데이터 스키마 정의, 공급업체 자료 정제와 단종 공고 파싱, 룰 기반 조달 위험 스크리닝, 텍스트 유사도와 지식 그래프를 쓴 대체 부품 추천, 붕괴 위험도와 조기경보, 위험 지도 대시보드와 오류 처리, 규격 검토와 승인 HITL 워크플로우, 공급망 차단 스트레스 시험, AAR 가중치 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 부품을 조기에 찾아 국산화 후보를 근거와 함께 제시하되, 규격 적합성 판단과 조달 승인은 담당자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 조달 프로세스 분석서, 스키마 정의서, UAT 보고서, 사용자 매뉴얼 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 위험 스크리닝 엔진, AI 추천 모델, 대시보드 UI, HITL 결재 FSM 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 타 부서 배포를 위한 원클릭 설치 스크립트(`setup.sh`) 및 최종 패키지

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# ============================================================================
# [산출물 자동 저장 유틸리티] 프로젝트 표준 디렉토리(docs/, src/, dist/) 자동 연계
# ============================================================================
def save_deliverable(filepath: str, content: str):
    """프로젝트 표준 deliverables 디렉토리에 산출물 파일을 자동 생성합니다."""
    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content.strip() + "\n")
    print(f"📦 [산출물 저장 완료] {filepath}")


---

## 🎯 미션 06. 부품 조달 위험 공급망 지도 및 대체선 추천 인터랙티브 대시보드 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

부품 조달 위험 공급망 지도 및 대체선 추천 대시보드를 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **부품 조달 위험 공급망 지도와 대체선 추천 대시보드에 입력 검증과 에러 핸들러를 붙여 깨진 입력에도 멈추지 않게 한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **국가와 부품 범주별 평균 위험을 보이는 공급망 지도, 리드타임과 지연률 산점도, 위험 상위 부품의 대체 후보 규격 적합성 패널을 가진 대시보드 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/utils/prc07_validator.py 및 에러 핸들러` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 사용자에게 알기 쉬운 에러 메시지 및 조치 가이드 제공

#### 미션 PASS 조건

- 정량 기준: 예외 상황 발생 시 크래시(Crash) 0건, 에러 로그 기록률 100%


- 진행 기준: 예외 주입 테스트 10종 통과 및 감사 로그 파일 생성


- 검증 방법: 비정상 데이터(결측, 특수문자, 대용량) 스트레스 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""조달 위험 공급망 지도(국가 x 부품 범주 위험 히트맵)와 대체선 추천 패널을 그리는 대시보드에 입력 검증기와 에러 핸들러를 붙이고, 깨진 입력 14종을 주입해 크래시 0건·에러 로그 기록률 100%·사용자용 조치 안내 문구를 확인한다."""
import csv
import io
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 무기체계 부품·공급망 데이터 생성기 (실제 체계·업체·국가·단가 정보 없음)
# 부품 100종(해외 조달 약 80%)의 공급망 속성과, 국내 부품 카탈로그(과거 승인된 국산 대체품 +
# 설명은 비슷하지만 규격이 맞지 않는 유사 후보)를 함께 만든다.
# ---------------------------------------------------------------------------
COUNTRIES = ["가상국가A", "가상국가B", "가상국가C", "국내"]
CATS = {
    "전원모듈": dict(words=["DC-DC", "컨버터", "절연형", "전원", "정출력"], syn={"컨버터": "변환장치", "절연형": "아이솔레이션", "정출력": "안정화"}, std="MIL-STD-704", iface="단자대"),
    "커넥터": dict(words=["원형", "커넥터", "방수형", "다핀", "나사체결"], syn={"커넥터": "접속자", "방수형": "밀폐형", "나사체결": "스크루결합"}, std="MIL-DTL-38999", iface="원형핀"),
    "유압밸브": dict(words=["유압", "솔레노이드", "밸브", "방향제어", "4포트"], syn={"밸브": "개폐장치", "방향제어": "유로전환", "솔레노이드": "전자식"}, std="SAE-AS5440", iface="플랜지"),
    "베어링": dict(words=["볼", "베어링", "고속", "밀봉형", "회전축"], syn={"베어링": "축받이", "밀봉형": "실드형", "고속": "고회전"}, std="ISO-492", iface="축결합"),
    "광학렌즈": dict(words=["적외선", "렌즈", "게르마늄", "반사방지", "집광"], syn={"렌즈": "광학창", "집광": "초점형성", "반사방지": "AR코팅"}, std="MIL-PRF-13830", iface="마운트"),
    "자이로센서": dict(words=["광섬유", "자이로", "각속도", "센서", "관성"], syn={"센서": "감지기", "각속도": "회전율", "관성": "INS용"}, std="IEEE-952", iface="직렬통신"),
    "RF증폭기": dict(words=["RF", "전력증폭기", "광대역", "모듈", "송신"], syn={"전력증폭기": "파워앰프", "송신": "출력단", "광대역": "와이드밴드"}, std="MIL-STD-461", iface="SMA"),
    "제어보드": dict(words=["FPGA", "제어보드", "임베디드", "연산", "입출력"], syn={"제어보드": "컨트롤카드", "연산": "신호처리", "임베디드": "내장형"}, std="VITA-46", iface="백플레인"),
}
V_OPTS, W_OPTS = [12, 24, 28], [25, 50, 100]
VENDORS = ["가상정밀(주)", "가상전자(주)", "가상유공압(주)", "가상광학(주)", "가상테크(주)", "가상시스템(주)"]


def describe(cat, words, s):
    # [TODO : 학생 실습 6] 부품 조달 위험 공급망 지도 및 대체선 추천 인터랙티브 대시보드 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def _sig(x):
    # [TODO : 학생 실습 6] 부품 조달 위험 공급망 지도 및 대체선 추천 인터랙티브 대시보드 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_parts(n=100, seed=7):
    """부품 목록(공급망 속성 포함)과 국내 카탈로그를 만든다. 반환: (parts DataFrame, catalog DataFrame)"""
    rng = np.random.default_rng(seed)
    cats, rows, cat_rows = list(CATS), [], []
    for i in range(n):
        cat = cats[i % len(cats)]; c = CATS[cat]
        spec = {"v": int(rng.choice(V_OPTS)), "w": int(rng.choice(W_OPTS)), "tmin": int(rng.choice([-55, -40])),
                "tmax": int(rng.choice([85, 125])), "size_mm": float(round(rng.uniform(20, 120), 1)), "std": c["std"]}
        words = [str(x) for x in rng.permutation(c["words"])[:4]]
        country = str(rng.choice(COUNTRIES, p=[0.35, 0.3, 0.15, 0.2]))
        overseas = country != "국내"
        n_sup = int(rng.choice([1, 2, 3], p=[0.45, 0.35, 0.2])) if overseas else int(rng.choice([1, 2, 3]))
        lead = float(round(rng.gamma(3, 45 if overseas else 25) + 20))
        eol = str(rng.choice(["정상", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
        crit = str(rng.choice(["A", "B", "C"], p=[0.3, 0.4, 0.3]))
        z = (-2.6 + 1.1 * (n_sup == 1) + 0.9 * overseas + 0.006 * (lead - 100) + 1.2 * (eol == "단종예고")
             + 2.0 * (eol == "단종") + rng.normal(0, 0.6))
        p_late = float(np.clip(_sig(z) * 0.8 + 0.05, 0, 0.9))
        late = int(rng.binomial(12, p_late))
        demand = int(rng.integers(20, 400))
        rows.append({"part_no": f"P-{i + 1:04d}", "category": cat, "desc": describe(cat, words, spec), **spec,
                     "iface": c["iface"], "country": country, "overseas": overseas, "n_suppliers": n_sup,
                     "lead_time_days": lead, "orders_12m": 12, "late_orders": late,
                     "avg_delay_days": float(round(late and rng.gamma(2, 12 + 20 * p_late), 1)),
                     "eol_status": eol, "ltb_months": int(rng.integers(1, 25)) if eol == "단종예고" else None,
                     "price_change_pct": float(round(rng.normal(4 + 12 * p_late, 8), 1)), "criticality": crit,
                     "annual_demand": demand, "stock_qty": int(rng.integers(0, demand)),
                     "defect_rate": float(round(abs(rng.normal(0.8 + 2 * p_late, 0.8)), 2)),
                     "export_license": bool(overseas and rng.random() < 0.35),
                     "disrupted_next12m": bool(rng.random() < _sig(z + 0.4))})
        if not overseas:
            continue
        # 과거 승인된 국산 대체품(정답): 동의어로 서술, 규격 호환(전력 여유 허용, 일부는 크기 차이가 커 장착 검토 필요)
        eq_words = [c["syn"].get(w, w) for w in words]
        hard = rng.random() < 0.15                                   # 서술이 많이 다른 사례
        if hard:
            eq_words = [c["syn"].get(w, w) for w in c["words"] if w not in words][:1] + eq_words[1:3]
        eq = {**spec, "w": spec["w"] if rng.random() < 0.7 else W_OPTS[min(W_OPTS.index(spec["w"]) + 1, 2)],
              "size_mm": float(round(spec["size_mm"] * (rng.uniform(1.11, 1.15) if rng.random() < 0.12 else rng.uniform(0.95, 1.05)), 1))}
        cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, eq_words, eq), **eq,
                         "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": f"P-{i + 1:04d}", "kind": "equivalent"})
        # 유사 오답 후보: 설명은 거의 같지만 전압이 다르거나(5V/48V) 상용 온도 등급(0~70C)
        for kind in ("volt_mismatch", "temp_mismatch"):
            d = dict(spec)
            if kind == "volt_mismatch":
                d["v"] = int(rng.choice([5, 48]))                           # 체계에서 쓰지 않는 전압
            else:
                d["tmin"], d["tmax"] = 0, 70
            d["size_mm"] = float(round(spec["size_mm"] * rng.uniform(0.95, 1.05), 1))
            cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, words, d), **d,
                             "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": None, "kind": kind})
    parts = pd.DataFrame(rows)
    catalog = pd.DataFrame(cat_rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    return parts, catalog
# ---------------------------------------------------------------------------

# ---- 대체 부품 추천 공통 함수: 텍스트 유사도 + 규격 적합성 + 지식 그래프 이웃 유사도 ----
WEIGHTS = {"text": 0.35, "spec": 0.45, "graph": 0.20}


def spec_fit(p, c):
    """규격 적합성 0~1: 6개 조건 충족 비율. 전압·온도 한계(필수 조건)를 못 맞추면 점수를 절반으로 깎는다."""
    conds = [p["v"] == c["v"], c["w"] >= p["w"], c["tmin"] <= p["tmin"], c["tmax"] >= p["tmax"],
             abs(c["size_mm"] - p["size_mm"]) / p["size_mm"] <= 0.10, p["iface"] == c["iface"]]
    s = sum(conds) / len(conds)
    return (s * 0.5 if not (conds[0] and conds[2] and conds[3]) else s), conds


def graph_nodes(row):
    """지식 그래프에서 부품 노드와 연결된 이웃 노드 집합(범주·표준·인터페이스·기능어·등급)."""
    words = set(row["desc"].split()[1:-4])
    grade = "군용등급" if row["tmin"] <= -40 else "상용등급"
    return {f"cat:{row['category']}", f"std:{row['std']}", f"if:{row['iface']}", f"grade:{grade}", f"v:{row['v']}"} | {f"w:{w}" for w in words}


def build_graph(catalog):
    """동의어 사전을 그래프 간선(기능어 = 동의어)으로 넣어 서술이 달라도 같은 기능 노드로 연결되게 한다."""
    syn = {f"w:{b}": f"w:{a}" for c in CATS.values() for a, b in c["syn"].items()}
    canon = lambda nodes: {syn.get(x, x) for x in nodes}
    return {"canon": canon, "nodes": [canon(graph_nodes(r)) for _, r in catalog.iterrows()]}


def graph_sim(g, pn, cn):
    """두 부품이 공유하는 이웃 노드 비율(Jaccard)."""
    pn = g["canon"](pn)
    return len(pn & cn) / max(len(pn | cn), 1)


def score_table(parts_df, catalog):
    """해외 부품 x 같은 범주 국내 후보 쌍마다 텍스트·규격·그래프 점수를 계산한 표(가중치 적용 전)."""
    vec = TfidfVectorizer(token_pattern=r"[^ ]+").fit(list(parts_df["desc"]) + list(catalog["desc"]))
    text = cosine_similarity(vec.transform(parts_df["desc"]), vec.transform(catalog["desc"]))
    g = build_graph(catalog)
    rows = []
    for i, (_, p) in enumerate(parts_df.iterrows()):
        pn = graph_nodes(p)
        for j, (_, c) in enumerate(catalog.iterrows()):
            if c["category"] != p["category"]:
                continue
            sf, conds = spec_fit(p, c)
            rows.append({"part_no": p["part_no"], "cand_no": c["cand_no"], "vendor": c["vendor"], "text": text[i, j],
                         "spec": sf, "graph": graph_sim(g, pn, g["nodes"][j]),
                         "unmet": ",".join(n for n, ok in zip(["전압", "전력", "저온", "고온", "크기", "인터페이스"], conds) if not ok) or "-",
                         "kind": c["kind"], "correct": c["approved_for"] == p["part_no"]})
    return pd.DataFrame(rows)


def recommend(table, weights=None, top=3):
    """가중합 점수로 부품별 상위 top개 후보를 고른다."""
    w = weights or WEIGHTS
    t = table.assign(score=w["text"] * table["text"] + w["spec"] * table["spec"] + w["graph"] * table["graph"])
    return t.sort_values(["part_no", "score"], ascending=[True, False]).groupby("part_no").head(top)


def hit_rate(recs, k=1):
    """과거 승인 이력(정답 대체품)이 상위 k개 안에 들어간 부품 비율."""
    return recs.groupby("part_no").apply(lambda d: d["correct"].head(k).any(), include_groups=False).mean()

NEED = {"part_no": str, "category": str, "country": str, "n_suppliers": float, "lead_time_days": float, "late_orders": float,
        "orders_12m": float, "eol_status": str, "annual_demand": float, "stock_qty": float, "desc": str, "v": float, "w": float,
        "tmin": float, "tmax": float, "size_mm": float, "std": str, "iface": str}
RANGE = {"n_suppliers": (1, 10), "lead_time_days": (1, 1000), "late_orders": (0, 100), "orders_12m": (1, 100), "annual_demand": (1, 100000), "stock_qty": (0, 100000)}
LOG = []


class UserError(Exception):
    """사용자에게 보여 줄 메시지와 조치 가이드를 함께 가진 예외."""

    def __init__(self, code, msg, action):
        super().__init__(msg); self.code, self.msg, self.action = code, msg, action


def log(level, code, detail):
    LOG.append({"seq": len(LOG) + 1, "level": level, "code": code, "detail": str(detail)[:80]})


def read_input(src):
    """입력(CSV 문자열 / JSON 문자열 / DataFrame)을 DataFrame으로 읽는다. 실패하면 UserError."""
    if src is None:
        raise UserError("E01", "입력이 비어 있습니다.", "파일을 다시 선택해 주세요.")
    if isinstance(src, pd.DataFrame):
        df = src.copy()
    elif isinstance(src, str):
        if not src.strip():
            raise UserError("E02", "빈 파일입니다.", "내용이 있는 파일인지 확인해 주세요.")
        try:
            df = pd.DataFrame(json.loads(src)) if src.lstrip()[0] in "[{" else pd.DataFrame(list(csv.DictReader(io.StringIO(src))))
        except Exception as e:
            raise UserError("E03", "파일 형식을 읽을 수 없습니다.", "CSV(쉼표 구분) 또는 JSON(목록) 형식으로 저장했는지 확인해 주세요.") from e
    else:
        raise UserError("E04", f"지원하지 않는 입력 형식입니다({type(src).__name__}).", "CSV 또는 JSON 파일을 사용해 주세요.")
    if df.empty:
        raise UserError("E02", "읽은 행이 0건입니다.", "머리글만 있는 파일이 아닌지 확인해 주세요.")
    return df


def validate_table(df):
    """열 점검 -> 행 단위 점검. 문제 행은 격리하고 나머지로 계속 진행한다. 반환: (정상 행, 격리 내역)"""
    miss = [c for c in NEED if c not in df.columns]
    if miss:
        raise UserError("E10", f"필수 열이 없습니다: {', '.join(miss[:4])}{' 외' if len(miss) > 4 else ''}", "열 이름이 양식(데이터 딕셔너리)과 같은지 확인해 주세요.")
    d = df.copy(); bad = {}
    for c, t in NEED.items():
        if t is float:
            conv = pd.to_numeric(d[c], errors="coerce")
            for i in d.index[conv.isna()]:
                bad.setdefault(i, f"{c}: 숫자가 아님/누락")
            d[c] = conv
    for c, (lo, hi) in RANGE.items():
        for i in d.index[(d[c] < lo) | (d[c] > hi)]:
            bad.setdefault(i, f"{c}: 허용 범위({lo}~{hi}) 밖")
    for i in d.index[d["country"].isin(COUNTRIES) == False]:
        bad.setdefault(i, "country: 허용되지 않은 값")
    for i in d.index[d["part_no"].duplicated(keep="first")]:
        bad.setdefault(i, "part_no: 중복")
    for i in d.index[~d["part_no"].astype(str).str.match(r"^P-\d{4}$")]:
        bad.setdefault(i, "part_no: 형식 오류")
    if len(bad) == len(d):
        raise UserError("E11", "사용할 수 있는 행이 없습니다.", "격리 사유를 확인해 값을 고친 뒤 다시 올려 주세요.")
    return d.drop(index=list(bad)), bad


def risk_points(d):
    rate = d["late_orders"] / d["orders_12m"]
    cover = d["stock_qty"] / d["annual_demand"] * 365 - d["lead_time_days"]
    return (30 * (d["n_suppliers"] == 1) + 25 * np.clip(rate, 0, 1) + 20 * (d["eol_status"] != "정상") + 15 * (cover < 0) + 10 * (d["country"] != "국내")).round(1)


def dashboard(df, catalog, draw=True):
    d, bad = validate_table(read_input(df))
    d = d.assign(risk=risk_points(d))
    heat = d.pivot_table(index="country", columns="category", values="risk", aggfunc="mean").round(1)
    over = d[d["country"] != "국내"].sort_values("risk", ascending=False).head(6)
    vec = TfidfVectorizer(token_pattern=r"[^ ]+").fit(list(d["desc"]) + list(catalog["desc"]))
    alt = []
    for _, p in over.iterrows():
        cs = catalog[catalog["category"] == p["category"]]
        if cs.empty:
            alt.append((p["part_no"], "후보 없음", 0.0)); continue
        fit = np.array([spec_fit(p, c)[0] for _, c in cs.iterrows()])
        sc = 0.45 * fit + 0.35 * cosine_similarity(vec.transform([p["desc"]]), vec.transform(cs["desc"]))[0]
        j = int(np.argmax(sc)); alt.append((p["part_no"], cs.iloc[j]["cand_no"], float(fit[j])))
    if draw:
        fig, ax = plt.subplots(1, 3, figsize=(13, 4))
        im = ax[0].imshow(heat.fillna(0).to_numpy(), cmap="Reds"); ax[0].set_xticks(range(heat.shape[1]), [f"c{i + 1}" for i in range(heat.shape[1])]); ax[0].set_yticks(range(heat.shape[0]), ["A", "B", "C", "domestic"][:heat.shape[0]])
        ax[0].set_title("Supply risk map (country x category)"); plt.colorbar(im, ax=ax[0])
        ax[1].scatter(d["lead_time_days"], d["late_orders"] / d["orders_12m"], c=d["risk"], cmap="Reds", s=25); ax[1].set_xlabel("lead time (days)"); ax[1].set_ylabel("late order rate"); ax[1].set_title("Lead time vs delay")
        ax[2].barh([a[0] for a in alt], [a[2] for a in alt], color="tab:blue"); ax[2].invert_yaxis(); ax[2].set_xlim(0, 1); ax[2].set_title("Alternative spec fit (top-6 risk)")
        plt.tight_layout(); plt.show()
    return heat, alt, bad


def safe_dashboard(case, src, catalog):
    """모든 예외를 잡아 크래시 없이 사용자 메시지·조치 안내를 돌려주고 에러 로그를 남긴다."""
    try:
        heat, alt, bad = dashboard(src, catalog, draw=False)
        for i, why in bad.items():
            log("WARN", "W01", f"{case}: 행 {i} 격리 - {why}")
        return {"case": case, "ok": True, "msg": f"표시 완료(격리 {len(bad)}행)" if bad else "표시 완료", "action": "-", "quarantined": len(bad)}
    except UserError as e:
        log("ERROR", e.code, f"{case}: {e.msg}")
        return {"case": case, "ok": False, "msg": e.msg, "action": e.action, "quarantined": 0}
    except Exception as e:                                   # 예상하지 못한 오류도 크래시 대신 안내로 변환
        log("ERROR", "E99", f"{case}: {type(e).__name__} {e}")
        return {"case": case, "ok": False, "msg": "예상하지 못한 오류가 발생했습니다.", "action": "오류 코드 E99와 입력 파일을 담당자에게 전달해 주세요.", "quarantined": 0}


parts, catalog = make_parts(100, seed=7)
base = parts.copy()
csv_ok = base.to_csv(index=False)

def mut(**kw):
    d = base.copy()
    for k, v in kw.items():
        d[k] = d[k].astype(object); d.loc[d.index[:3], k] = v
    return d

cases = [
    ("정상 입력", base), ("정상 CSV 문자열", csv_ok), ("입력 없음(None)", None), ("빈 문자열", "  "), ("깨진 JSON", '[{"part_no": "P-0001", '),
    ("지원하지 않는 형식(정수)", 12345), ("머리글만 있는 CSV", ",".join(base.columns) + "\n"), ("필수 열 누락", base.drop(columns=["lead_time_days", "desc"])),
    ("숫자 열에 문자", mut(lead_time_days="미정")), ("음수 리드타임", mut(lead_time_days=-5)), ("공급선 수 결측", mut(n_suppliers=np.nan)),
    ("허용되지 않은 국가", mut(country="미상")), ("부품번호 중복·형식 오류", mut(part_no="P-9")), ("연간 소요량 0(나눗셈 위험)", mut(annual_demand=0)),
    ("극단값(1e12)", mut(stock_qty=1e12)), ("전부 불량 행", base.assign(lead_time_days=-1)),
]
res = [safe_dashboard(n, s, catalog) for n, s in cases]
print(f"[에러 핸들러 시험] 입력 {len(res)}종 실행")
for r in res:
    print(f"  {'OK ' if r['ok'] else '안내'} {r['case']:22s} -> {r['msg']}" + (f" | 조치: {r['action']}" if r["action"] != "-" else ""))
crash = 0                                                    # safe_dashboard 밖으로 예외가 새 나간 건수(위에서 모두 포착)
n_err = sum(not r["ok"] for r in res)
logged_err = sum(l["level"] == "ERROR" for l in LOG)
print(f"\n크래시 {crash}건(목표 0건: {'달성' if crash == 0 else '미달'}), 오류 {n_err}건 중 로그 기록 {logged_err}건 -> 기록률 {logged_err / max(n_err, 1):.0%}(목표 100%: {'달성' if logged_err == n_err else '미달'})")
print(f"행 단위 격리 경고 {sum(l['level'] == 'WARN' for l in LOG)}건 기록, 안내 문구에 조치 가이드 포함 {sum(r['action'] != '-' for r in res if not r['ok'])}/{n_err}건")
print("\n[에러 로그 앞 5건]")
print(pd.DataFrame(LOG).head(5).to_string(index=False))

# 정상 입력으로 대시보드 그리기
heat, alt, bad = dashboard(base, catalog)
print("\n[공급망 지도] 국가 x 범주 평균 위험 점수(상위 위험 국가 행)")
print(heat.loc[[c for c in COUNTRIES if c in heat.index]].fillna(0).round(0).astype(int).to_string())
print("\n[대체선 추천 패널] 위험 상위 해외 부품별 국산 후보와 규격 적합성")
for a in alt:
    print(f"  {a[0]} -> {a[1]} (규격 적합성 {a[2]:.2f})")


<details>
<summary><b>💡 [참고] 미션 06 예상 실행 결과 (Output)</b></summary>

```text
[에러 핸들러 시험] 입력 16종 실행
  OK  정상 입력                  -> 표시 완료
  OK  정상 CSV 문자열             -> 표시 완료
  안내 입력 없음(None)            -> 입력이 비어 있습니다. | 조치: 파일을 다시 선택해 주세요.
  안내 빈 문자열                  -> 빈 파일입니다. | 조치: 내용이 있는 파일인지 확인해 주세요.
  안내 깨진 JSON                -> 파일 형식을 읽을 수 없습니다. | 조치: CSV(쉼표 구분) 또는 JSON(목록) 형식으로 저장했는지 확인해 주세요.
  안내 지원하지 않는 형식(정수)         -> 지원하지 않는 입력 형식입니다(int). | 조치: CSV 또는 JSON 파일을 사용해 주세요.
  안내 머리글만 있는 CSV            -> 읽은 행이 0건입니다. | 조치: 머리글만 있는 파일이 아닌지 확인해 주세요.
  안내 필수 열 누락                -> 필수 열이 없습니다: lead_time_days, desc | 조치: 열 이름이 양식(데이터 딕셔너리)과 같은지 확인해 주세요.
  OK  숫자 열에 문자               -> 표시 완료(격리 3행)
  OK  음수 리드타임                -> 표시 완료(격리 3행)
  OK  공급선 수 결측               -> 표시 완료(격리 3행)
  OK  허용되지 않은 국가             -> 표시 완료(격리 3행)
  OK  부품번호 중복·형식 오류          -> 표시 완료(격리 3행)
  OK  연간 소요량 0(나눗셈 위험)       -> 표시 완료(격리 3행)
  OK  극단값(1e12)              -> 표시 완료(격리 3행)
  안내 전부 불량 행                -> 사용할 수 있는 행이 없습니다. | 조치: 격리 사유를 확인해 값을 고친 뒤 다시 올려 주세요.

크래시 0건(목표 0건: 달성), 오류 7건 중 로그 기록 7건 -> 기록률 100%(목표 100%: 달성)
행 단위 격리 경고 21건 기록, 안내 문구에 조치 가이드 포함 7/7건

[에러 로그 앞 5건]
 seq level code                                 detail
   1 ERROR  E01              입력 없음(None): 입력이 비어 있습니다.
   2 ERROR  E02                        빈 문자열: 빈 파일입니다.
   3 ERROR  E03             깨진 JSON: 파일 형식을 읽을 수 없습니다.
   4 ERROR  E04 지원하지 않는 형식(정수): 지원하지 않는 입력 형식입니다(int).
   5 ERROR  E02              머리글만 있는 CSV: 읽은 행이 0건입니다.

[공급망 지도] 국가 x 범주 평균 위험 점수(상위 위험 국가 행)
category  RF증폭기  광학렌즈  베어링  유압밸브  자이로센서  전원모듈  제어보드  커넥터
country                                                 
가상국가A        39    58   48    41     44    36    46   69
가상국가B        42    49   75    54     51    68    57   30
가상국가C        14    63   39     0     16     0    71   29
국내            0    19    4    38     28     2    28   42

[대체선 추천 패널] 위험 상위 해외 부품별 국산 후보와 규격 적합성
  P-0080 -> D-0205 (규격 적합성 1.00)
  P-0066 -> D-0172 (규격 적합성 1.00)
... (이하 생략)
```
</details>
